# 🎭 23 · Separating *what* a molecule is from *how* it is posed: Harmony and beyond
### Autoencoders, variational autoencoders, spatial VAEs and Siamese disentanglement of content from pose — on 2D views of the molecular zoo

**BiologyToResearch** · Part E — Unsupervised & geometric deep learning (2020–2026) · Module 23 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/23_disentangling_identity_pose.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU is enough (≈ 4–6 min to run everything); a GPU makes training ~3× faster but is optional |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, scikit-learn, PyTorch |
| **Prerequisites** | Module 05 (PyTorch training loops), Module 21 (spatial transformer, `affine_grid`/`grid_sample`), Module 22 (clustering without labels). Variational autoencoders are introduced from scratch here. |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | **Harmony** (Uddin, Howe, Zeng & Xu, CVPR 2022); **3D spatial VAE** (Yi *et al.*, arXiv 2021); successors: Uddin, Armouti & Xu (ICCV 2025) and DUIL (NeurIPS 2026) — *titles only*, plus a related ICLR 2026 submission — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments are small, controlled *toy versions* of published methods: 2D images instead of 3D subtomograms, three
> molecules, tiny networks and a few minutes of CPU training. They illustrate the core ideas; they do **not**
> reproduce the papers' numbers. Two successor papers could not be read while writing; they are described by title
> only. **Nothing is stored in the repository:** molecules are real Protein Data Bank (PDB) models, downloaded the
> first time a notebook needs them and cached on your computer (offline, code-built stand-ins of the same size and
> symmetry are used instead); all images and data are simulated by code.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why the *pose* of a particle (rotation, shift) dominates pixel-by-pixel similarity, and **measure** it
   by comparing nearest-neighbour classification on raw and on pose-corrected images.
2. **Explain** an autoencoder and a variational autoencoder (VAE) — encoder, latent code, decoder, reparameterization
   trick, KL divergence and the ELBO — and **verify** the Gaussian KL formula numerically.
3. **Implement** a 2D *spatial VAE* whose decoder draws the molecule at rotated and shifted coordinates, and traverse its
   latent variables.
4. **Implement** Harmony-lite: encoder → (pose, content); spatial-transformer pose normalisation; a decoder that sees only
   content; a Siamese copy under a random known transform; and Harmony's exact loss.
5. **Evaluate** disentanglement with the class predictivity $P(c\,|\,z)$ (k-nearest neighbours on held-out data) and pose
   recovery in degrees, and **run** the ablation that removes the Siamese branch.
6. **Describe** what the lab's successor works aim at — and what we could *not* verify — and **try** multiple-choice
   (winner-takes-all) learning.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | check the computer, load the toolkit and three molecules | 5 min |
| 1 · Pose hides identity | build a dataset of randomly posed views; nearest neighbours with and without pose | 20 min |
| 2 · Autoencoders → VAEs | the ELBO, the reparameterization trick, a first VAE and its tangled latent space | 30 min |
| 3 · Spatial VAE | make rotation and translation explicit latent variables | 25 min |
| 4 · Harmony | Siamese disentanglement of content from transformation; evaluation; ablation | 40 min |
| 5 · Beyond Harmony | successors, and a multiple-choice (winner-takes-all) toy | 15 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 15 min |

## 🧫 The biological question

A cell is full of molecular machines caught at random orientations. A **20S proteasome** (the cell's protein
shredder) and a **singly capped proteasome** — the same barrel with a regulatory cap bolted onto one end — differ in
*composition*. A **ribosome** changes *conformation* as it moves along a messenger RNA. Those differences are the
biology. But in a tomogram every copy is also turned and shifted in its own random way — the **pose** — and pose changes
the picture far more than the cap or the conformation does.

If we want a computer to *discover* compositions and conformations without labels (Modules 15 and 22), we need a
description of each particle in which **"what it is" (content)** is stored separately from **"how it is posed"
(transformation)**. Learning such a *disentangled* description is the goal of the lab's **Harmony** (CVPR 2022) and of
its successors. In this notebook you will build one, piece by piece.

---
## 0 · Environment setup

Run the cells in order. The first one only reports what computer you are on.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook runs on a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet (the next cell installs it).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

The helpers below are copied verbatim from [`course_toolkit.py`](../course_toolkit.py) so that this notebook runs on its
own: building zoo maps (Module 00), preparing them (Module 01), the 2D rotation `rotate_image` (Module 02) and the
Pearson correlation (Module 04). Just run the cell.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_EXTRA_ENTRIES = {"5a1a": dict(name="β-galactosidase", organism="E. coli", mass_kDa=465, symmetry="D2")}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
def standin_atoms(pid):
    """Atomic model (dict of arrays) of the code-built stand-in for zoo entry `pid` (Å, centred at the origin).

    Used when the real entry cannot be downloaded: ideal α-helices arranged with the size, mass and symmetry of the
    PDB entry. Deterministic (the random numbers are seeded by the entry's name)."""
    pid = pid.lower()
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    return atoms


def pdb_cache_dir():
    """Folder (outside the repository) where downloaded PDB files are kept."""
    return os.environ.get("BIO2RESEARCH_PDB_CACHE") or os.path.join(os.path.expanduser("~"), ".cache",
                                                                    "biology_to_research", "pdb")


def _pdb_urls(pid, kind):
    """Download addresses of one PDB file: kind = 'entry' or 'assembly1' (mmCIF) or 'pdb' (legacy PDB format)."""
    pid = pid.lower()
    mid = pid[1:3]
    if kind == "entry":
        return [f"https://files.rcsb.org/download/{pid.upper()}.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/mmCIF/{mid}/{pid}.cif.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/{pid}.cif"]
    if kind == "assembly1":
        return [f"https://files.rcsb.org/download/{pid.upper()}-assembly1.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/assemblies/mmCIF/divided/{mid}/{pid}-assembly1.cif.gz"]
    if kind == "pdb":
        return [f"https://files.rcsb.org/download/{pid.upper()}.pdb.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/pdb/{mid}/pdb{pid}.ent.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{pid}.ent"]
    raise ValueError(kind)


def fetch_pdb(pid, kind="entry", timeout=60):
    """Text of a PDB file, downloaded once and cached outside the repository. Returns (text, where_from), or
    (None, reason) when no mirror can be reached (or BIO2RESEARCH_STANDINS is set)."""
    import gzip
    import urllib.request
    name = {"entry": f"{pid.lower()}.cif.gz", "assembly1": f"{pid.lower()}-assembly1.cif.gz",
            "pdb": f"{pid.lower()}.pdb.gz"}[kind]
    path = os.path.join(pdb_cache_dir(), name)
    if os.path.exists(path):
        return gzip.decompress(open(path, "rb").read()).decode("utf-8", "replace"), f"cached copy ({path})"
    if os.environ.get("BIO2RESEARCH_STANDINS"):
        return None, "BIO2RESEARCH_STANDINS is set"
    reason = "no mirror tried"
    for url in _pdb_urls(pid, kind):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                raw = response.read()
            text = (gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw).decode("utf-8", "replace")
            if ("_atom_site." not in text) and ("\nATOM  " not in text):
                raise ValueError("no atoms in the downloaded file")
            os.makedirs(pdb_cache_dir(), exist_ok=True)
            with open(path, "wb") as f:
                f.write(gzip.compress(text.encode()))
            return text, url
        except Exception as e:  # noqa: BLE001 — any failure: try the next mirror
            reason = f"{type(e).__name__}: {str(e)[:120]}"
    return None, reason


def _cif_tokens(line):
    """Split one line of an mmCIF loop into values (quoted values may contain spaces)."""
    if "'" not in line and '"' not in line:
        return line.split()
    import re
    toks = re.findall(r"'(?:[^']|'(?=\S))*'|\"(?:[^\"]|\"(?=\S))*\"|\S+", line)
    return [t[1:-1] if len(t) > 1 and t[0] == t[-1] and t[0] in "'\"" else t for t in toks]


_WATER = {"HOH", "DOD", "WAT", "H2O"}
def read_mmcif_atoms(text):
    """Atoms of the first model in an mmCIF file → dict(xyz, name, element, resname, chain, resnum).

    Reads the `_atom_site` loop; skips water and hydrogen, and keeps only the first of alternative atom locations."""
    lines = text.splitlines()
    i = next(k for k, line in enumerate(lines) if line.startswith("_atom_site."))
    cols = []
    while i < len(lines) and lines[i].startswith("_atom_site."):
        cols.append(lines[i].split()[0][len("_atom_site."):])
        i += 1
    c = {name: k for k, name in enumerate(cols)}
    get = lambda *names: next((c[n] for n in names if n in c), None)   # noqa: E731
    ix, iy, iz = c["Cartn_x"], c["Cartn_y"], c["Cartn_z"]
    i_el, i_name = get("type_symbol"), get("auth_atom_id", "label_atom_id")
    i_res, i_ch = get("auth_comp_id", "label_comp_id"), get("auth_asym_id", "label_asym_id")
    i_num, i_alt, i_mod = get("auth_seq_id", "label_seq_id"), get("label_alt_id"), get("pdbx_PDB_model_num")
    xyz, name, element, resname, chain, resnum = [], [], [], [], [], []
    first_model, buf = None, []
    for line in lines[i:]:
        if not line or line.startswith(("#", "_", "loop_", "data_")):
            break
        buf += _cif_tokens(line)
        while len(buf) >= len(cols):
            row, buf = buf[:len(cols)], buf[len(cols):]
            if i_mod is not None:
                first_model = row[i_mod] if first_model is None else first_model
                if row[i_mod] != first_model:
                    continue
            el = row[i_el].upper() if i_el is not None else row[i_name][:1].upper()
            if el in ("H", "D") or row[i_res] in _WATER:
                continue
            if i_alt is not None and row[i_alt] not in (".", "?", "A", "1"):
                continue
            xyz.append((float(row[ix]), float(row[iy]), float(row[iz])))
            name.append(row[i_name])
            element.append(el.capitalize() if len(el) > 1 else el)
            resname.append(row[i_res])
            chain.append(row[i_ch])
            num = row[i_num] if i_num is not None else "0"
            resnum.append(int(num) if num.lstrip("-").isdigit() else 0)
    return dict(xyz=np.array(xyz, np.float32), name=np.array(name), element=np.array(element),
                resname=np.array(resname), chain=np.array(chain), resnum=np.array(resnum))


_ATOMIC_MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "P": 30.974, "S": 32.06, "Se": 78.97, "Mg": 24.305,
                "Zn": 65.38, "Fe": 55.845, "Mn": 54.938, "K": 39.098, "Na": 22.99, "Cl": 35.45, "Ca": 40.078}
def model_mass_kDa(atoms):
    """Mass of an atomic model without hydrogens, plus ~8 % for the missing hydrogens (kDa)."""
    return 1.08 * sum(_ATOMIC_MASS.get(e, 12.0) for e in atoms["element"]) / 1000.0


def canonical_frame(atoms):
    """Centre a model and rotate it onto its principal axes — longest extent along x, shortest along z — with the
    signs fixed by the skewness of the atom distribution. A proteasome barrel then lies along x and a ring
    (RuBisCO, p97) lies in the x–y plane, like the stand-ins."""
    xyz = atoms["xyz"].astype(np.float64)
    X = xyz - xyz.mean(0)
    evecs = np.linalg.eigh(np.cov(X.T))[1][:, ::-1]                 # columns: principal axes, largest first
    Y = X @ evecs
    sign = np.sign(np.sum(Y ** 3, axis=0))
    sign[sign == 0] = 1.0
    Y *= sign
    if np.linalg.det(evecs * sign) < 0:                             # keep a proper rotation (no mirror image)
        Y[:, 2] *= -1
    out = dict(atoms)
    out["xyz"] = Y.astype(np.float32)
    return out


def pdb_atoms(pid, verbose=False):
    """Real atomic model of a PDB entry, in the canonical frame. Returns (atoms, description), or (None, reason).

    Uses the deposited entry unless its mass is far from ZOO's reference mass, in which case the first biological
    assembly is tried and the closer of the two kept (e.g. one ribosome rather than the two of a crystal)."""
    pid = pid.lower()
    ref = ZOO.get(pid, {}).get("mass_kDa") or _EXTRA_ENTRIES.get(pid, {}).get("mass_kDa")
    candidates, reason = [], ""
    for kind in ("entry", "assembly1"):
        if candidates and (ref is None or abs(np.log(model_mass_kDa(candidates[0][0]) / ref)) < np.log(1.3)):
            break
        text, where = fetch_pdb(pid, kind)
        if text is None:
            reason = where
            continue
        try:
            atoms = read_mmcif_atoms(text)
        except (StopIteration, KeyError, ValueError) as e:
            reason = f"could not read the file ({type(e).__name__})"
            continue
        if len(atoms["xyz"]):
            label = "deposited entry" if kind == "entry" else "biological assembly 1"
            candidates.append((atoms, f"PDB {pid.upper()} ({label}; {len(atoms['xyz']):,} atoms; from {where})"))
    if not candidates:
        return None, reason
    atoms, desc = min(candidates, key=lambda a: abs(np.log(model_mass_kDa(a[0]) / ref)) if ref else 0.0)
    if verbose:
        print(desc)
    return canonical_frame(atoms), desc


_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays, Å, centred) of zoo entry `pid`: the real PDB model when it can be downloaded, the
    code-built stand-in otherwise. The source is recorded in `atoms_source(pid)`."""
    pid = pid.lower()
    if pid not in _ATOMS_CACHE:
        atoms, desc = pdb_atoms(pid)
        if atoms is None:
            atoms, desc = standin_atoms(pid), f"code-built stand-in modelled on PDB {pid.upper()} (download failed: {desc})"
        _ATOMS_CACHE[pid] = (atoms, desc)
    return _ATOMS_CACHE[pid][0]


def atoms_source(pid):
    """Where the atoms of zoo entry `pid` came from (a real PDB file or the offline stand-in)."""
    zoo_atoms(pid)
    return _ATOMS_CACHE[pid.lower()][1]


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0, "Mg": 12.0, "Zn": 30.0, "Fe": 26.0, "K": 19.0,
      "Na": 11.0, "Cl": 17.0, "Ca": 20.0, "Mn": 25.0, "Se": 34.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of a zoo entry, painted from its atoms (`zoo_atoms`)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, atoms_source(pid)


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def load_zoo(ids=("4v4r", "3j9i", "6utj", "1bxn"), size: int = 32, sigma: float = 1.0,
             verbose: bool = True) -> dict:
    """Build and prepare several zoo maps. Returns {pdb_id: volume(size³)}."""
    zoo = {}
    for pid in ids:
        raw, vs, src = zoo_map(pid)
        zoo[pid] = prepare_map(raw, size=size, sigma=sigma)
        if verbose:
            print(f"  {pid.upper():5s} {ZOO.get(pid, {}).get('name', ''):28s} raw {raw.shape} @ {vs:.0f} Å/voxel  ← {src}")
    return zoo


def rotate_image(img: np.ndarray, angle_rad: float, shift_xy=(0, 0), order: int = 1) -> np.ndarray:
    """2D analogue of :func:`rotate_volume` (counter-clockwise in the (x, y) plane)."""
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    R = np.array([[c, -s], [s, c]])
    R_idx = R[::-1, ::-1]                      # (x,y) -> (y,x)
    ctr = (np.array(img.shape) - 1) / 2.0
    t_idx = np.asarray(shift_xy, float)[::-1]
    M = R_idx.T
    return ndimage.affine_transform(img, M, offset=ctr - M @ (ctr + t_idx), order=order, mode="constant")


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))
print("🧰 toolkit loaded")

### Data: three molecules seen from the side

We work in **2D** so that everything trains in minutes on a CPU: each molecule is represented by one projection (a side
view, the image an electron microscope would record if the molecule lay on its side). Pose is then an in-plane rotation
angle $\theta$ and a shift $(t_x, t_y)$ — the 2D analogue of the three angles and three shifts of a subtomogram.

| class | PDB | molecule | what makes it different |
|---|---|---|---|
| 0 | 4V4R | 70S ribosome | large, two unequal subunits |
| 1 | 3J9I | 20S proteasome | a barrel of four stacked rings |
| 2 | 6UTJ | singly capped proteasome | the same barrel **plus a cap** at one end — a *compositional* variant of class 1 |

In [ ]:
# ── Load the cast and make one canonical side view per molecule ──
CAST = [("4v4r", 1, "70S ribosome"), ("3j9i", 0, "20S proteasome"), ("6utj", 1, "capped proteasome")]
zoo = load_zoo([pid for pid, _, _ in CAST], size=32)
canon = []
for pid, axis, _ in CAST:
    view = zoo[pid].sum(axis=axis)                 # projection = sum of density along one axis
    canon.append(view / view.max())                # scale each view to a maximum of 1
canon = np.stack(canon).astype(np.float32)         # [3, 32, 32]
CLASS_NAMES = [name for _, _, name in CAST]
SHORT = ["ribosome", "20S", "capped"]                 # short labels for figures
N_PIX = canon.shape[-1]

fig, axes = plt.subplots(1, 3, figsize=(6.6, 2.4))
for ax, im, (pid, _, name) in zip(axes, canon, CAST):
    ax.imshow(im); ax.set_title(f"{pid.upper()} · {name}", fontsize=8); ax.axis("off")
fig.suptitle("Canonical side views (32 × 32 px, 1 nm per pixel)", fontsize=9)
plt.tight_layout(); plt.show()

---
## 📍 Where we are

In **Module 21** you built networks that *align* two particles (Gum-Net) with a differentiable spatial transformer, and in
**Module 22** you sorted particles into classes without labels (DISCA). Both had to fight the same enemy: pose. This
module learns a **representation** in which pose and content live in different variables, so that sorting,
discovering and aligning become easier — the idea behind Harmony.

---
## 1 · Pose hides identity

### 1.1 A differentiable spatial transformer (recap)

From Module 21: PyTorch's `affine_grid` builds, for every output pixel $p$, the input location it should be read from,
and `grid_sample` reads it with bilinear interpolation. For a rotation $R(\theta)$ about the image centre $c$ followed by
a shift $t$:

$$x_{\text{out}}(p) \;=\; x_{\text{in}}\big(R(\theta)^{-1}(p - c - t) + c\big).$$

*In words:* to know what lands at pixel $p$, undo the shift, undo the rotation, and look up the input there. Because
every step is a smooth formula, a network can learn $\theta$ and $t$ by gradient descent. We check that our function
agrees with the toolkit's `rotate_image`.

In [ ]:
# ── 1.1 Spatial transformer: rotate about the centre, then translate ──
def spatial_transform(img, theta, shift):
    """Rotate images about their centre by ``theta`` (radians), then translate them by ``shift`` (pixels).

    img: tensor [B, 1, H, W]; theta: tensor [B]; shift: tensor [B, 2] holding (t_x, t_y).
    Same convention as the toolkit's ``rotate_image``: out(p) = img(R⁻¹(p − c − t) + c).
    Built from ``affine_grid`` + ``grid_sample``, so gradients flow back to ``theta`` and ``shift``.
    """
    B, _, H, W = img.shape
    c, s = torch.cos(theta), torch.sin(theta)
    Rt = torch.stack([torch.stack([c, s], 1), torch.stack([-s, c], 1)], 1)    # Rᵀ = R⁻¹, shape [B, 2, 2]
    t_norm = torch.stack([shift[:, 0] * 2 / W, shift[:, 1] * 2 / H], 1)       # pixels → [−1, 1] units
    b = -(Rt @ t_norm.unsqueeze(-1)).squeeze(-1)
    grid = F.affine_grid(torch.cat([Rt, b.unsqueeze(-1)], 2), list(img.shape), align_corners=False)
    return F.grid_sample(img, grid, mode="bilinear", padding_mode="zeros", align_corners=False)


x0 = torch.from_numpy(canon[:1]).unsqueeze(1)
angle = 0.5                                                   # radians (≈ 29°)
ours = spatial_transform(x0, torch.tensor([angle]), torch.zeros(1, 2))
back = spatial_transform(ours, torch.tensor([-angle]), torch.zeros(1, 2))
r_tool = pearson(ours[0, 0].numpy(), rotate_image(canon[0], angle))
print(f"agreement with the toolkit's rotate_image: r = {r_tool:.5f}")
print(f"rotate by +θ then by −θ: mean squared difference {float(((back - x0) ** 2).mean()):.1e} "
      "(bilinear interpolation blurs a little)")
assert r_tool > 0.999, "the transformer should match rotate_image"

### 1.2 A dataset of randomly posed, noisy views

Each training image is a canonical view turned by a random angle $\theta \sim U(-180°, 180°)$, shifted by up to ±3 px
in $x$ and $y$, plus white Gaussian noise. We use a signal-to-noise ratio **SNR = 1** (variance ratio, the course
convention) — gentle compared with real data (≈ 0.01–0.1 per subtomogram) so that our tiny networks learn in minutes;
you will lower it in a ✏️ exercise. A **circular mask** (zero outside a disc) is applied, as is common for particle
images, so that rotating an image never creates new corners.

We generate a **training set** (500 images per class) and an independent, **held-out test set** (200 per class). The
class labels are kept *only for evaluation* — none of the models in Sections 2–4 ever sees them.

In [ ]:
# ── 1.2 Build training and test sets ──
SNR = 1.0
yy, xx = np.mgrid[:N_PIX, :N_PIX] - (N_PIX - 1) / 2
CIRCLE = torch.from_numpy((np.hypot(xx, yy) <= N_PIX / 2 - 0.5).astype(np.float32))[None, None]


def make_pose_dataset(n_per_class, rng, noise_sd=None, snr=SNR, max_shift=3.0):
    """Randomly rotated (−180°…180°) and shifted (±max_shift px) copies of every canonical view, plus noise.

    Returns a dict: x (noisy, circle-masked), clean, y (class), theta (radians), shift (px) and noise_sd.
    If ``noise_sd`` is None it is set so that var(clean)/var(noise) = snr.
    """
    base = torch.from_numpy(canon).unsqueeze(1)
    y = torch.arange(len(canon)).repeat_interleave(n_per_class)
    theta = torch.from_numpy(rng.uniform(-np.pi, np.pi, len(y))).float()
    shift = torch.from_numpy(rng.uniform(-max_shift, max_shift, (len(y), 2))).float()
    clean = spatial_transform(base[y], theta, shift)
    noise_sd = float(clean.std()) / np.sqrt(snr) if noise_sd is None else noise_sd
    noisy = (clean + noise_sd * torch.from_numpy(rng.standard_normal(clean.shape).astype(np.float32))) * CIRCLE
    return dict(x=noisy, clean=clean, y=y, theta=theta, shift=shift, noise_sd=noise_sd)


train_set = make_pose_dataset(500, np.random.default_rng(SEED))
test_set = make_pose_dataset(200, np.random.default_rng(SEED + 1), noise_sd=train_set["noise_sd"])
NOISE_VAR = train_set["noise_sd"] ** 2
X_TRAIN, X_TEST = train_set["x"].to(DEVICE), test_set["x"].to(DEVICE)
CIRCLE = CIRCLE.to(DEVICE)
Y_TRAIN, Y_TEST = train_set["y"].numpy(), test_set["y"].numpy()
TH_TRAIN, TH_TEST = train_set["theta"].numpy(), test_set["theta"].numpy()
print(f"train {tuple(X_TRAIN.shape)}  test {tuple(X_TEST.shape)}  noise σ = {train_set['noise_sd']:.3f} (SNR {SNR})")

fig, axes = plt.subplots(2, 9, figsize=(9, 2.4))
for j, i in enumerate(np.linspace(0, len(Y_TEST) - 1, 9).astype(int)):
    axes[0, j].imshow(test_set["clean"][i, 0]); axes[1, j].imshow(test_set["x"][i, 0])
    axes[0, j].set_title(f"class {Y_TEST[i]}, {np.degrees(TH_TEST[i]):.0f}°", fontsize=7)
    axes[0, j].axis("off"); axes[1, j].axis("off")
axes[0, 0].text(-6, 16, "clean", rotation=90, va="center", fontsize=8)
axes[1, 0].text(-6, 16, "noisy", rotation=90, va="center", fontsize=8)
plt.tight_layout(); plt.show()

### 1.3 Experiment: do nearest neighbours in pixel space find the same molecule?

A simple test of a representation: label a few images per class, and classify every test image by its **nearest
labelled neighbour** (1-NN; distance = Euclidean distance between the feature vectors). We compare two features:

- **raw pixels** of the noisy images;
- **oracle-aligned pixels**: the same images after undoing their *true* rotation and shift (only possible in
  simulation — this is the best a perfect pose estimator could do).

In [ ]:
# ── 1.3 Few-label nearest-neighbour accuracy: raw vs pose-corrected pixels ──
def oracle_align(d):
    """Undo the TRUE pose of every image in dataset ``d``: translate back, then rotate back (simulation only)."""
    n = len(d["x"])
    x = spatial_transform(d["x"], torch.zeros(n), -d["shift"])
    return spatial_transform(x, -d["theta"], torch.zeros(n, 2))


def few_label_accuracy(f_train, y_train, f_test, y_test, n_per_class, repeats=10, seed=SEED):
    """Mean 1-nearest-neighbour test accuracy when only ``n_per_class`` random training examples per class are labelled."""
    g = np.random.default_rng(seed)
    accs = []
    for _ in range(repeats):
        sel = np.concatenate([g.choice(np.where(y_train == c)[0], n_per_class, replace=False)
                              for c in np.unique(y_train)])
        accs.append(KNeighborsClassifier(1).fit(f_train[sel], y_train[sel]).score(f_test, y_test))
    return float(np.mean(accs))


raw_tr, raw_te = train_set["x"].flatten(1).numpy(), test_set["x"].flatten(1).numpy()
ali_tr, ali_te = oracle_align(train_set).flatten(1).numpy(), oracle_align(test_set).flatten(1).numpy()
n_labels = [1, 3, 10, 30, 100, 300]
acc_raw = [few_label_accuracy(raw_tr, Y_TRAIN, raw_te, Y_TEST, n) for n in n_labels]
acc_ali = [few_label_accuracy(ali_tr, Y_TRAIN, ali_te, Y_TEST, n) for n in n_labels]
for n, a, b in zip(n_labels, acc_raw, acc_ali):
    print(f"{n:4d} labels/class: raw pixels {a:.2f}   oracle-aligned {b:.2f}")

pca = PCA(2).fit(raw_tr)
p = pca.transform(raw_te)
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2))
axes[0].semilogx(n_labels, acc_raw, "o-", label="raw pixels")
axes[0].semilogx(n_labels, acc_ali, "s-", label="oracle-aligned pixels")
axes[0].axhline(1 / 3, color="gray", ls=":", label="chance")
axes[0].set_xlabel("labelled examples per class"); axes[0].set_ylabel("1-NN test accuracy")
axes[0].set_title("Pose is a nuisance"); axes[0].legend(fontsize=7)
for c in range(3):
    axes[1].scatter(*p[Y_TEST == c].T, s=4, label=CLASS_NAMES[c])
axes[1].legend(fontsize=7, markerscale=3); axes[1].set_title("PCA of raw pixels, coloured by class")
sc = axes[2].scatter(*p.T, s=4, c=np.degrees(TH_TEST) % 180, cmap="twilight")
plt.colorbar(sc, ax=axes[2], label="angle mod 180°"); axes[2].set_title("…and coloured by angle")
for ax in axes[1:]:
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With raw pixels, a nearest neighbour is mostly "an image turned the same way", whatever the
molecule — so you need hundreds of labelled examples per class (enough to cover every pose) before 1-NN works. Remove the
pose and **a handful of labels suffices**. The PCA panels tell the same story: the two strongest directions of variation in
raw pixels are explained by the *angle* (smooth colour gradient), not by the molecule (mixed colours).

💡 This is the **nuisance-variable problem**: pose is a large source of variation that carries no biological information.
We want a learned code $z$ that behaves like the *right* curve without being told the true poses.

---
## 2 · From autoencoders to variational autoencoders (VAEs)

**Autoencoder.** Squeeze each image through a narrow bottleneck and ask the network to rebuild it:

$$x \;\xrightarrow{\ \text{encoder } f\ }\; z \in \mathbb{R}^{d}\;\xrightarrow{\ \text{decoder } g\ }\; \hat{x} = g(z),
\qquad \text{loss} = \lVert x - \hat{x} \rVert^2 .$$

*In words:* the **latent code** $z$ (here just $d = 2$ numbers) must summarise everything needed to redraw the image, so it
becomes a compact description of the particle. Problem: nothing organises this code — nearby codes need not mean similar
images, and we cannot sample new, sensible codes.

**Variational autoencoder (VAE).** Kingma & Welling (2014) made the code *probabilistic*:

1. The encoder outputs a **distribution** over codes, a Gaussian $q(z\,|\,x) = \mathcal{N}(\mu(x), \sigma^2(x))$ —
   "this image is probably described by codes around $\mu$, give or take $\sigma$".
2. We **sample** a code from it. To let gradients pass through the random sampling we write the sample as
   $z = \mu + \sigma \cdot \varepsilon$ with $\varepsilon \sim \mathcal{N}(0, 1)$ — the **reparameterization trick**:
   the randomness sits in $\varepsilon$, while $\mu$ and $\sigma$ stay ordinary differentiable numbers.
3. The decoder turns $z$ back into an image.
4. The loss has two parts — together the negative **evidence lower bound (ELBO)**:

$$-\text{ELBO}(x) \;=\; \underbrace{\frac{1}{2s^2}\,\lVert x - g(z)\rVert^2}_{\text{reconstruction}}
\;+\; \underbrace{\mathrm{KL}\big(q(z\,|\,x)\,\Vert\,\mathcal{N}(0, I)\big)}_{\text{keep codes tidy}} \;+\;\text{const.}$$

*In words:* redraw the image well (the first term is the Gaussian log-likelihood of the image, with pixel-noise variance
$s^2$), but keep every image's code distribution close to a standard bell curve so that the latent space is smooth and
filled without holes. Maximising the ELBO maximises a lower bound on the probability the model assigns to the data.

### 🧮 The Kullback–Leibler (KL) divergence of two Gaussians

$\mathrm{KL}(q\,\Vert\,p) = \mathbb{E}_{z\sim q}[\log q(z) - \log p(z)]$ measures how different distribution $q$ is from
$p$ (zero only when they are identical; not symmetric). For one-dimensional Gaussians it has a closed form,

$$\mathrm{KL}\big(\mathcal{N}(\mu_1,\sigma_1^2)\,\Vert\,\mathcal{N}(\mu_2,\sigma_2^2)\big)
= \log\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1-\mu_2)^2}{2\sigma_2^2} - \frac12 ,$$

summed over dimensions for diagonal Gaussians. *In words:* you pay for shifting the mean (in units of the reference
spread) and for making the spread too narrow or too wide. Let us check the formula against brute-force sampling.

In [ ]:
# ── 2.1 The KL divergence of two diagonal Gaussians: formula vs Monte Carlo ──
def kl_gauss(mu1, logvar1, mu2, logvar2):
    """KL( N(mu1, exp(logvar1)) ‖ N(mu2, exp(logvar2)) ) for diagonal Gaussians, summed over the last dimension."""
    return 0.5 * (logvar2 - logvar1 + (logvar1.exp() + (mu1 - mu2) ** 2) / logvar2.exp() - 1).sum(-1)


mu_q, lv_q = torch.tensor([[1.0, -0.5]]), torch.tensor([[-1.0, 0.3]])
mu_p, lv_p = torch.zeros(1, 2), torch.zeros(1, 2)
z = mu_q + torch.exp(0.5 * lv_q) * torch.randn(200_000, 2)                      # samples from q
log_q = torch.distributions.Normal(mu_q, torch.exp(0.5 * lv_q)).log_prob(z).sum(1)
log_p = torch.distributions.Normal(mu_p, torch.exp(0.5 * lv_p)).log_prob(z).sum(1)
kl_formula, kl_mc = kl_gauss(mu_q, lv_q, mu_p, lv_p).item(), (log_q - log_p).mean().item()
print(f"KL by formula = {kl_formula:.4f}   KL by Monte Carlo (200,000 samples) = {kl_mc:.4f}")
assert abs(kl_formula - kl_mc) < 0.02, "formula and sampling should agree"

### 2.2 A first VAE on our posed molecules

Encoder and decoder are small **multilayer perceptrons (MLPs)** — the same kind of network Harmony's released 2D code
uses — with a 2-dimensional code so that we can plot it. One generic training loop serves every model in this notebook.

In [ ]:
# ── 2.2 Building blocks: MLP encoder/decoder, reparameterization, a generic training loop ──
class MLPEncoder(nn.Module):
    """Image [B, 1, 32, 32] → ``n_out`` numbers, through two hidden tanh layers."""

    def __init__(self, n_out, n_pix=N_PIX, hidden=(256, 128)):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(n_pix * n_pix, hidden[0]), nn.Tanh(),
                                 nn.Linear(hidden[0], hidden[1]), nn.Tanh(), nn.Linear(hidden[1], n_out))

    def forward(self, x):
        return self.net(x)


class MLPDecoder(nn.Module):
    """Latent vector [B, z_dim] → image [B, 1, 32, 32] (the mirror image of the encoder)."""

    def __init__(self, z_dim, n_pix=N_PIX, hidden=(128, 256)):
        super().__init__()
        self.n_pix = n_pix
        self.net = nn.Sequential(nn.Linear(z_dim, hidden[0]), nn.Tanh(), nn.Linear(hidden[0], hidden[1]), nn.Tanh(),
                                 nn.Linear(hidden[1], n_pix * n_pix))

    def forward(self, z):
        return self.net(z).view(-1, 1, self.n_pix, self.n_pix)


def reparameterize(mu, logvar):
    """z = μ + σ·ε with ε ~ N(0, 1): a random sample that gradients can flow through (the reparameterization trick)."""
    return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)


def train(model, loss_fn, x_train, steps, batch=64, lr=1e-3, seed=SEED, log_every=250, name="model"):
    """Adam on random mini-batches of ``x_train``; ``loss_fn(model, x)`` returns (loss, dict of numbers to log)."""
    model.to(DEVICE).train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    gen = torch.Generator().manual_seed(seed)
    torch.manual_seed(seed)                    # makes the random ε of the VAE repeatable as well
    history, t0 = [], time.time()
    for step in range(1, steps + 1):
        idx = torch.randint(0, len(x_train), (batch,), generator=gen)
        loss, logs = loss_fn(model, x_train[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        history.append(dict(loss=loss.item(), **logs))
        if step == 1 or step % log_every == 0:
            print(f"  [{name}] step {step:4d}  loss {loss.item():8.1f}  "
                  + "  ".join(f"{k} {v:7.2f}" for k, v in logs.items()))
    model.eval()
    print(f"  [{name}] {steps} steps in {time.time() - t0:.0f} s")
    return history

In [ ]:
# ── 2.3 Train a plain VAE (2-D latent code) ──
class VAE(nn.Module):
    """A plain variational autoencoder with a ``z_dim``-dimensional Gaussian latent code."""

    def __init__(self, z_dim=2):
        super().__init__()
        self.z_dim = z_dim
        self.enc, self.dec = MLPEncoder(2 * z_dim), MLPDecoder(z_dim)

    def encode(self, x):
        """Return (μ, log σ²) of q(z|x)."""
        h = self.enc(x)
        return h[:, :self.z_dim], h[:, self.z_dim:]

    def content_mean(self, x):
        """Posterior mean μ of the code (used for evaluation)."""
        return self.encode(x)[0]

    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.dec(reparameterize(mu, logvar)), mu, logvar


def vae_loss(model, x):
    """−ELBO per image = SSE/(2 s²) + KL(q(z|x) ‖ N(0, I)), with s² the known pixel-noise variance."""
    recon, mu, logvar = model(x)
    rec = (((recon - x) * CIRCLE) ** 2).sum((1, 2, 3)) / (2 * NOISE_VAR)
    kl = kl_gauss(mu, logvar, torch.zeros_like(mu), torch.zeros_like(logvar))
    return (rec + kl).mean(), dict(rec=rec.mean().item(), kl=kl.mean().item())


torch.manual_seed(SEED)
vae = VAE(z_dim=2)
hist_vae = train(vae, vae_loss, X_TRAIN, steps=1000, name="VAE")

In [ ]:
# ── 2.4 Look at the VAE: reconstructions, latent space and class predictivity ──
def predictivity(f_train, f_test, k=5):
    """P(c|z): accuracy of a k-nearest-neighbour classifier fitted on training codes, tested on held-out codes."""
    return float(KNeighborsClassifier(k).fit(f_train, Y_TRAIN).score(f_test, Y_TEST))


def plot_latent(z, labels, angles, title):
    """Two scatter plots of a 2-D code: coloured by class, and by the true angle (mod 180°)."""
    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.3))
    for c in range(3):
        axes[0].scatter(*z[labels == c, :2].T, s=4, label=CLASS_NAMES[c])
    axes[0].legend(fontsize=7, markerscale=3); axes[0].set_title(f"{title}: coloured by class")
    sc = axes[1].scatter(*z[:, :2].T, s=4, c=np.degrees(angles) % 180, cmap="twilight")
    plt.colorbar(sc, ax=axes[1], label="true angle mod 180°"); axes[1].set_title(f"{title}: coloured by angle")
    for ax in axes:
        ax.set_xlabel("z₁"); ax.set_ylabel("z₂")
    plt.tight_layout(); plt.show()


@torch.no_grad()
def codes(model, x):
    """Posterior means of the content code for every image (no sampling), as a NumPy array."""
    return model.content_mean(x).cpu().numpy()


RESULTS = {}                                    # method → (P(c|z) with all training codes, 1-NN acc. with 10 labels)
RESULTS["raw pixels"] = (predictivity(raw_tr, raw_te), few_label_accuracy(raw_tr, Y_TRAIN, raw_te, Y_TEST, 10))
z_vae_tr, z_vae_te = codes(vae, X_TRAIN), codes(vae, X_TEST)
RESULTS["plain VAE"] = (predictivity(z_vae_tr, z_vae_te), few_label_accuracy(z_vae_tr, Y_TRAIN, z_vae_te, Y_TEST, 10))

show = [0, 150, 220, 380, 420, 590]
with torch.no_grad():
    recon = vae.dec(vae.content_mean(X_TEST[show])).cpu()
fig, axes = plt.subplots(2, len(show), figsize=(7.5, 2.7))
for j, i in enumerate(show):
    axes[0, j].imshow(test_set["x"][i, 0]); axes[1, j].imshow(recon[j, 0])
    axes[0, j].axis("off"); axes[1, j].axis("off")
axes[0, 0].set_title("test input", fontsize=8, loc="left"); axes[1, 0].set_title("VAE reconstruction", fontsize=8, loc="left")
plt.tight_layout(); plt.show()
plot_latent(z_vae_te, Y_TEST, TH_TEST, "plain VAE")
for name, (p_cz, few) in RESULTS.items():
    print(f"{name:12s}  P(c|z) = {p_cz:.2f}   10-label 1-NN = {few:.2f}")

🔍 **What just happened?** The VAE learned to redraw the particles (blurry but recognisable — the blur is the price of
a 2-number code and of the noise). Its code is organised mostly by **orientation**: the angle colouring varies smoothly
across the latent plane, while the three molecules overlap. With only two numbers to spend, the network spent them on
the biggest source of variation — the pose — which is exactly what we do *not* want.

✏️ **Your turn.** Change `VAE(z_dim=2)` to `VAE(z_dim=8)` in cell 2.3 and re-run 2.3–2.4 (the scatter plots then show
only the first two dimensions). Does $P(c\,|\,z)$ improve? Is the pose information gone, or just spread over more numbers?

---
## 3 · Spatial VAE: make rotation and translation explicit latent variables

**Idea** (Bepler *et al.* 2019 for 2D; Yi *et al.* 2021 extended it to 3D subtomograms). Make the decoder a function of
**position**: a small network $g_\phi$ answers *"what is the intensity at coordinate $u$ for a molecule with content
$z$?"* To draw that molecule rotated by $\theta$ and shifted by $t$, ask the same network at transformed coordinates:

$$\log p(x \mid \theta, t, z) \;=\; \sum_{i\in\text{pixels}} \log \mathcal{N}\big(x_i \;\big|\; g_\phi(R(\theta)\,u_i + t,\; z),\ s^2\big).$$

*In words:* every pixel $u_i$ of the image is looked up in the molecule's own coordinate frame. Rotation and shift are
no longer something the network has to *learn to draw* — they are built into the geometry, so they can live in their own
latent variables $(\theta, t)$ and leave $z$ for the content. The encoder now outputs a Gaussian posterior for
$(\theta, t_x, t_y, z_1, z_2)$, each with its own prior: $\theta \sim \mathcal{N}(0, \sigma_\theta^2)$,
$t \sim \mathcal{N}(0, \sigma_t^2)$, $z \sim \mathcal{N}(0, I)$, and

$$\text{ELBO} = \mathbb{E}_q\big[\log p(x\mid\theta,t,z)\big] - \mathrm{KL}_\theta - \mathrm{KL}_t - \mathrm{KL}_z .$$

Yi *et al.* used prior standard deviations $\sigma_\theta = \pi/2$ and $\sigma_t = 0.14$ (coordinates scaled to
$[-1, 1]$). Our particles take *any* in-plane angle, so we use $\sigma_\theta = \pi$. A practical bonus of a coordinate
decoder: we can train on a **random subset of pixels** each step (here 256 of 1,024), which makes it 4× cheaper.

In [ ]:
# ── 3.1 A 2D spatial VAE ──
g1 = torch.linspace(-1 + 1 / N_PIX, 1 - 1 / N_PIX, N_PIX)                   # pixel centres in [−1, 1]
GY, GX = torch.meshgrid(g1, g1, indexing="ij")
COORDS = torch.stack([GX.ravel(), GY.ravel()], 1).to(DEVICE)                  # [1024, 2] as (x, y), row-major
PRIOR_SD = torch.tensor([np.pi, 0.14, 0.14, 1.0, 1.0], device=DEVICE)        # σ_θ, σ_tx, σ_ty, σ_z1, σ_z2


class SpatialVAE(nn.Module):
    """2D spatial VAE: the decoder evaluates intensity at transformed coordinates R(θ)·u + t, given content z."""

    def __init__(self, z_dim=2, hidden=64):
        super().__init__()
        self.z_dim = z_dim
        self.enc = MLPEncoder(2 * (3 + z_dim))           # μ and log σ² for (θ, t_x, t_y, z_1 … z_d)
        self.coord_in = nn.Linear(2, hidden)
        self.z_in = nn.Linear(z_dim, hidden, bias=False)
        self.body = nn.Sequential(nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def encode(self, x):
        """Return (μ, log σ²), each [B, 3 + z_dim] ordered as (θ, t_x, t_y, z…)."""
        h = self.enc(x)
        n = 3 + self.z_dim
        return h[:, :n], h[:, n:]

    def content_mean(self, x):
        return self.encode(x)[0][:, 3:]

    def decode(self, z, theta, t, coords=COORDS):
        """Intensities at ``coords`` [P, 2] for every (z, θ, t) in the batch → [B, P]."""
        c, s = torch.cos(theta), torch.sin(theta)
        R = torch.stack([torch.stack([c, -s], 1), torch.stack([s, c], 1)], 1)       # [B, 2, 2]
        moved = coords.unsqueeze(0) @ R.transpose(1, 2) + t.unsqueeze(1)             # R(θ)·u + t for every pixel
        return self.body(self.coord_in(moved) + self.z_in(z).unsqueeze(1)).squeeze(-1)

    def forward(self, x, coords=COORDS):
        mu, logvar = self.encode(x)
        sample = reparameterize(mu, logvar)
        return self.decode(sample[:, 3:], sample[:, 0], sample[:, 1:3], coords), mu, logvar


def svae_loss(model, x, n_pix=256):
    """−ELBO: SSE/(2 s²) on a random subset of ``n_pix`` pixels (rescaled to the full image) + KL_θ + KL_t + KL_z."""
    sel = torch.randperm(COORDS.shape[0], device=DEVICE)[:n_pix]
    recon, mu, logvar = model(x, COORDS[sel])
    err = (recon - x.flatten(1)[:, sel]) * CIRCLE.flatten()[sel]
    rec = (err ** 2).sum(1) * (COORDS.shape[0] / n_pix) / (2 * NOISE_VAR)
    kl = kl_gauss(mu, logvar, torch.zeros_like(mu), (2 * torch.log(PRIOR_SD)).expand_as(mu))
    return (rec + kl).mean(), dict(rec=rec.mean().item(), kl=kl.mean().item())


torch.manual_seed(SEED)
svae = SpatialVAE(z_dim=2)
hist_svae = train(svae, svae_loss, X_TRAIN, steps=800, batch=32, name="spatial VAE")

### 3.2 Walking through the spatial VAE's latent variables

Because $\theta$ enters the decoder only through the coordinates, changing $\theta$ must produce a **pure rotation** —
by construction. Changing $z$ is where the network decides what changes. We use the class labels *only* to choose
where to start the walk (the average code of each class).

In [ ]:
# ── 3.2 Latent traversals: θ rotates; z changes the molecule ──
z_svae_tr, z_svae_te = codes(svae, X_TRAIN), codes(svae, X_TEST)
class_means = torch.tensor(np.stack([z_svae_tr[Y_TRAIN == c].mean(0) for c in range(3)]), device=DEVICE).float()
with torch.no_grad():
    angles = torch.linspace(-np.pi / 2, np.pi / 2, 7, device=DEVICE)
    row_theta = svae.decode(class_means[:1].repeat(7, 1), angles, torch.zeros(7, 2, device=DEVICE)).cpu()
    drawn = svae.decode(class_means, torch.zeros(3, device=DEVICE), torch.zeros(3, 2, device=DEVICE)).cpu()
fig, axes = plt.subplots(2, 7, figsize=(8.4, 2.9))
for j in range(7):
    axes[0, j].imshow(row_theta[j].view(N_PIX, N_PIX))
    axes[0, j].set_title(f"θ = {np.degrees(angles[j].item()):.0f}°", fontsize=7)
for j, c in enumerate(range(3)):
    axes[1, j].imshow(drawn[c].view(N_PIX, N_PIX)); axes[1, j].set_title(f"mean z of\n{SHORT[c]}", fontsize=7)
lim = float((drawn - drawn[1]).abs().max())
for j, (c, lab) in enumerate([(0, "ribosome − 20S"), (2, "capped − 20S")]):
    axes[1, 4 + j].imshow((drawn[c] - drawn[1]).view(N_PIX, N_PIX), cmap="RdBu_r", vmin=-lim, vmax=lim)
    axes[1, 4 + j].set_title(lab, fontsize=7)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("Spatial VAE — top: change only θ (ribosome code);  bottom: change only z (θ = 0) and differences", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.3 Spatial VAE: latent space, predictivity and pose recovery ──
def pose_error_deg(theta_pred, theta_true, labels, period_deg=360.0):
    """Absolute error (degrees, modulo ``period_deg``) of a recovered pose.

    A model may pick ANY canonical orientation for each class, so a correct prediction satisfies
    θ̂ + θ_true = δ_c (a per-class constant). We estimate δ_c as the circular mean of θ̂ + θ_true within each
    class and return every particle's angular deviation from it.
    """
    scale = 360.0 / period_deg
    err = np.zeros(len(theta_pred))
    for c in np.unique(labels):
        m = labels == c
        r = (theta_pred[m] + theta_true[m]) * scale
        delta = np.angle(np.mean(np.exp(1j * r)))
        err[m] = np.abs(np.angle(np.exp(1j * (r - delta)))) / scale
    return np.degrees(err)


with torch.no_grad():
    th_svae = svae.encode(X_TEST)[0][:, 0].cpu().numpy()
RESULTS["spatial VAE"] = (predictivity(z_svae_tr, z_svae_te), few_label_accuracy(z_svae_tr, Y_TRAIN, z_svae_te, Y_TEST, 10))
plot_latent(z_svae_te, Y_TEST, TH_TEST, "spatial VAE")
for period in (360, 180):
    e = pose_error_deg(th_svae, TH_TEST, Y_TEST, period)
    print(f"spatial VAE pose error (mod {period}°): median {np.median(e):5.1f}°, within 15°: {np.mean(e < 15):.0%}")
print(f"spatial VAE  P(c|z) = {RESULTS['spatial VAE'][0]:.2f}   10-label 1-NN = {RESULTS['spatial VAE'][1]:.2f}")

🔍 **What just happened?** Turning $\theta$ (top row of 3.2) rotates the drawing exactly — by construction. Changing $z$
(bottom row) changes the drawn molecule only subtly: our 64-unit coordinate decoder, trained for under a minute, draws
blurry, similar blobs (the red/blue difference maps show where the codes differ: length and width). Yet the content code
**already separates the molecules** far better than the plain VAE's, and the angle colouring no longer organises the latent
plane — rotation now lives in $\theta$. The pose
estimate is good only **modulo 180°** — many particles are recognised "upside down". Side views of a proteasome barrel
really do look the same after a half turn, and for the ribosome this is a classic trap of rotation inference: the encoder
must guess the angle from the image alone, with nothing but the reconstruction error to correct it.

⚠️ Spatial VAEs are sensitive to the rotation prior and to local minima; a real 3D subtomogram adds two more angles and
the missing wedge. Yi *et al.* report their evidence mainly through ELBO values and latent traversals.

### 📄 Paper connection — 3D spatial VAE for cryo-ET

**Yi, Pang, Zhang, Zeng & Xu (2021).** *Disentangling semantic features of macromolecules in Cryo-Electron Tomography.*
[arXiv:2106.14192](https://arxiv.org/abs/2106.14192) (authors as listed in the lab's bibliography; venue not verified).
- **Question it answers:** can a VAE separate a subtomogram's rotation and translation from its remaining ("semantic")
  structural factors?
- **What the paper does:** an encoder gives Gaussian posteriors over rotation $\Theta$ (yaw, pitch, roll), translation
  $\Delta t$ and an unstructured code $z$; a coordinate-conditioned decoder computes
  $\log p_g(V|\Theta,\Delta t,z)=\sum_i \log p_g(v_i\,|\,x_iR(\Theta)+\Delta t,\,z)$ with
  $R(\Theta)=R(\theta_y)R(\theta_p)R(\theta_r)$; priors $\sigma_\Theta=\pi/2$, $\sigma_{\Delta t}=0.14$; loss = ELBO with
  three KL terms. 3D-SpVAE sums an MLP on $z$ and an MLP on coordinates; 3D-SpVAE++ uses a CNN decoder. No explicit
  missing-wedge handling is described.
- **What our toy keeps:** coordinate decoder, explicit $(\theta, t)$ latents with Gaussian priors, three-KL ELBO.
- **What it leaves out:** 3D, three Euler angles, the CNN decoder (++), real data, the paper's data sets
  (8 PDB structures at SNR 0.01–0.1; real "Noble single-particle" and RNA-polymerase subtomograms).
- **Result reported:** on real data 3D-SpVAE++ reaches a higher ELBO than a vanilla 3D VAE (e.g. −39.45 vs −43.95 on the
  RNA-polymerase set) and latent traversals change rotation and translation largely independently; the authors' evidence
  for disentanglement is mainly qualitative.
- **Code:** not in AITom.

---
## 4 · Harmony: Siamese disentanglement of content from transformation

Harmony (Uddin, Howe, Zeng & Xu, CVPR 2022) keeps the explicit pose variables but changes **how the network is taught**
to use them. Four ingredients:

1. **Encoder** $f(x) = (k_x, \psi_x)$: a pose $k_x = (\theta, t_x, t_y)$ and the parameters
   $\psi_x = (\mu, \log\sigma^2)$ of a Gaussian **content** code $z \sim \mathcal{N}(\mu, \mathrm{diag}\,\sigma^2)$.
2. **Spatial transformer** $T$: the predicted pose is applied **to the input itself**, giving a pose-normalised image
   $x_k = T(x; k_x)$.
3. **Decoder** $g$ sees **only** $z$ and must redraw $x_k$: $x_z = g(z)$. Since $x_k$ is (ideally) always in the same
   canonical pose, $z$ only has to describe content.
4. **Siamese copy**: a copy $x'$ of the same image under a **random, known** transformation goes through the same
   (weight-shared) encoder, transformer and decoder. Whatever pose $x'$ was in, its normalised version $x'_{k'}$ must
   equal $x_k$, and the two content distributions must agree.

The loss (the paper's equation, the released code's implementation):

$$\mathcal{L}(x, x') \;=\; \gamma\Big[D^{(1)}(x_z, x_k) + D^{(2)}(x_k, x'_{k'}) + D^{(3)}(x'_{k'}, x'_{z'})\Big]
\;+\; D_{\mathrm{KL}}\big(P_{\psi}\,\Vert\,P_{\psi'}\big).$$

*In words:* each branch must redraw its own pose-normalised image from content alone ($D^{(1)}, D^{(3)}$); the two
pose-normalised images must coincide ($D^{(2)}$ — this is what forces $T$ to undo the pose *consistently*); and the two
content posteriors must agree (KL). Each $D$ is a sum of squared errors (SSE). The weight $\gamma \propto M/N$
($M$ training images, $N$ batch size); the code uses $\gamma = \lceil M/(1000\,N) \rceil$. The authors deliberately **omit** a
direct term $D^{(4)}(x_z, x'_{z'})$, which they found hindered escaping local minima. Note there is **no** KL to a
standard normal prior here — the only KL compares the two branches.

In [ ]:
# ── 4.1 Harmony at a glance (diagram) ──
def draw_harmony_diagram():
    """Schematic of Harmony's two weight-shared branches and its four loss terms."""
    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    box = dict(boxstyle="round,pad=0.3", fc="#eef3fb", ec="#4a6fa5")
    img = dict(boxstyle="round,pad=0.3", fc="#fff6e0", ec="#c08a00")
    loss = dict(boxstyle="round,pad=0.2", fc="#fde8e8", ec="#c0392b")

    def arrow(a, b, rad=0.0, color="k"):
        ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="->", color=color, lw=1.2,
                                                        connectionstyle=f"arc3,rad={rad}"))
    for y0, prime in [(3.0, ""), (1.0, "′")]:
        up = 1 if prime == "" else -1
        ax.text(0.6, y0, f"x{prime}", ha="center", va="center", bbox=img, fontsize=10)
        ax.text(2.1, y0, "encoder f\n(shared)", ha="center", va="center", bbox=box, fontsize=8)
        ax.text(3.7, y0 + 0.5 * up, f"pose k{prime}=(θ,t)", ha="center", va="center", fontsize=8)
        ax.text(3.7, y0 - 0.5 * up, f"ψ{prime}=(μ, log σ²)", ha="center", va="center", fontsize=8)
        ax.text(5.4, y0 + 0.5 * up, f"transformer\nT(x{prime}; k{prime})", ha="center", va="center", bbox=box, fontsize=8)
        ax.text(5.2, y0 - 0.5 * up, f"z{prime} ~ N(μ, σ²)", ha="center", va="center", fontsize=8)
        ax.text(6.5, y0 - 0.5 * up, "decoder g\n(shared)", ha="center", va="center", bbox=box, fontsize=8)
        ax.text(7.9, y0 + 0.5 * up, f"x{prime}_k{prime}", ha="center", va="center", bbox=img, fontsize=9)
        ax.text(7.9, y0 - 0.5 * up, f"x{prime}_z{prime}", ha="center", va="center", bbox=img, fontsize=9)
        arrow((0.85, y0), (1.55, y0)); arrow((2.65, y0), (3.05, y0 + 0.45 * up)); arrow((2.65, y0), (3.05, y0 - 0.45 * up))
        arrow((4.35, y0 + 0.5 * up), (4.75, y0 + 0.5 * up)); arrow((4.45, y0 - 0.5 * up), (4.6, y0 - 0.5 * up))
        arrow((5.85, y0 - 0.5 * up), (6.0, y0 - 0.5 * up)); arrow((6.05, y0 + 0.5 * up), (7.5, y0 + 0.5 * up))
        arrow((7.0, y0 - 0.5 * up), (7.5, y0 - 0.5 * up))
        arrow((0.6, y0 + 0.25 * up), (5.0, y0 + 0.75 * up), rad=-0.15 * up, color="#7f8c8d")
        ax.plot([8.3, 8.5, 8.5, 8.3], [y0 + 0.5, y0 + 0.5, y0 - 0.5, y0 - 0.5], color="#c0392b", lw=1)
        ax.text(8.75, y0, "D⁽¹⁾" if prime == "" else "D⁽³⁾", ha="center", va="center", bbox=loss, fontsize=9)
    ax.annotate("", xy=(0.6, 1.3), xytext=(0.6, 2.7), arrowprops=dict(arrowstyle="->", lw=1.5, color="#8e44ad"))
    ax.text(0.75, 2.0, "random known\ntransform\nθ_r ~ U(−90°, 90°)\nt_r ~ U(−3, 3) px", fontsize=7, va="center", color="#8e44ad")
    ax.text(3.7, 2.0, "KL(P_ψ ‖ P_ψ′)", ha="center", va="center", bbox=loss, fontsize=9)
    ax.plot([8.3, 9.4, 9.4, 8.3], [3.5, 3.5, 0.5, 0.5], color="#c0392b", lw=1, ls="--")
    ax.text(9.4, 2.0, "D⁽²⁾", ha="center", va="center", bbox=loss, fontsize=9)
    ax.text(9.55, 3.85, "grey arrows: the input image itself is transformed", fontsize=7, ha="right", color="#7f8c8d")
    ax.set_xlim(0, 9.8); ax.set_ylim(-0.1, 4.1); ax.axis("off")
    ax.set_title("Harmony: two weight-shared branches, four loss terms (no direct D⁽⁴⁾(x_z, x′_z′))", fontsize=10)
    plt.show()


draw_harmony_diagram()

In [ ]:
# ── 4.2 Harmony-lite: model and loss ──
MAX_SHIFT = 4.0     # px; shifts are bounded with tanh, as the paper describes (the released 2D code leaves them unbounded)
GAMMA = math.ceil(len(X_TRAIN) / (1000 * 64))     # γ = ceil(M / (1000·N)), the released code's default
print("γ =", GAMMA)


class Harmony(nn.Module):
    """Harmony-lite (2D). Encoder → pose k = (θ, t) and content posterior ψ = (μ, log σ²); the spatial transformer
    pose-normalises the INPUT with the predicted pose; the decoder sees only the content code z."""

    def __init__(self, z_dim=2):
        super().__init__()
        self.z_dim = z_dim
        self.enc = MLPEncoder(3 + 2 * z_dim)        # [θ, t_x, t_y, μ…, log σ²…] — the code's output layout
        self.dec = MLPDecoder(z_dim)

    def encode(self, x):
        """Return θ (radians), t (pixels, [B, 2]), μ and log σ² of the content code."""
        phi = self.enc(x)
        theta, shift = phi[:, 0], MAX_SHIFT * torch.tanh(phi[:, 1:3])
        return theta, shift, phi[:, 3:3 + self.z_dim], phi[:, 3 + self.z_dim:]

    def content_mean(self, x):
        return self.encode(x)[2]

    def forward(self, x):
        theta, shift, mu, logvar = self.encode(x)
        x_k = spatial_transform(x, theta, shift)            # pose-normalised input
        x_z = self.dec(reparameterize(mu, logvar))          # redrawn from content only
        return x_z, x_k, mu, logvar


def harmony_loss(model, x, siamese=True):
    """γ[SSE(x_z, x_k) + SSE(x_k, x′_k′) + SSE(x′_k′, x′_z′)]/n + mean KL(P_ψ ‖ P_ψ′)/dim  (structure of the released code).

    With ``siamese=False`` only γ·SSE(x_z, x_k)/n remains — the ablation of Section 4.5.
    """
    n = x.shape[0]
    x_z, x_k, mu, logvar = model(x)
    d1 = ((x_z - x_k) ** 2).sum() / n
    if not siamese:
        return GAMMA * d1, dict(d1=d1.item())
    with torch.no_grad():                                          # the random KNOWN transform of the Siamese copy
        theta_r = (torch.rand(n, device=x.device) - 0.5) * np.pi    # U(−90°, 90°)
        shift_r = (torch.rand(n, 2, device=x.device) - 0.5) * 6.0   # U(−3, 3) px
        x_prime = spatial_transform(x, theta_r, shift_r)
    xp_z, xp_k, mu_p, logvar_p = model(x_prime)
    d2 = ((x_k - xp_k) ** 2).sum() / n
    d3 = ((xp_k - xp_z) ** 2).sum() / n
    kl = kl_gauss(mu, logvar, mu_p, logvar_p).mean() / model.z_dim
    return GAMMA * (d1 + d2 + d3) + kl, dict(d1=d1.item(), d2=d2.item(), d3=d3.item(), kl=kl.item())

### 4.3 Train Harmony-lite

1,000 steps of Adam (learning rate $10^{-3}$; the paper's code uses $10^{-4}$ and many more epochs). Watch the three
SSE terms fall: $D^{(2)}$ shrinking means the two branches agree on the canonical pose.

In [ ]:
# ── 4.3 Training ──
torch.manual_seed(SEED)
harmony = Harmony(z_dim=2)
hist_h = train(harmony, harmony_loss, X_TRAIN, steps=1000, name="Harmony")

fig, ax = plt.subplots(figsize=(5.6, 3.0))
for key in ["d1", "d2", "d3"]:
    ax.plot(np.convolve([h[key] for h in hist_h], np.ones(25) / 25, mode="valid"), label=key.replace("d", "D⁽") + "⁾")
ax.set_xlabel("training step"); ax.set_ylabel("SSE per image (running mean)"); ax.set_yscale("log")
ax.set_title("Harmony-lite training curves"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.4 What Harmony sees: input x, pose-normalised x_k, and the content-only drawing x_z ──
show = [0, 70, 140, 200, 270, 340, 400, 470, 540]
with torch.no_grad():
    th_s, sh_s, mu_s, _ = harmony.encode(X_TEST[show])
    xk_s = spatial_transform(X_TEST[show], th_s, sh_s).cpu()
    xz_s = harmony.dec(mu_s).cpu()
fig, axes = plt.subplots(3, len(show), figsize=(9, 3.4))
for j, i in enumerate(show):
    for r, im in enumerate([test_set["x"][i, 0], xk_s[j, 0], xz_s[j, 0]]):
        axes[r, j].imshow(im); axes[r, j].axis("off")
    axes[0, j].set_title(SHORT[Y_TEST[i]], fontsize=7)
for r, lab in enumerate(["input x", "x_k = T(x; k)", "x_z = g(μ)"]):
    axes[r, 0].text(-4, 16, lab, rotation=90, va="center", ha="right", fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The middle row is the encoder's own undoing of the pose: particles of the same kind are
turned into (nearly) the same orientation — the network invented a canonical pose per class without ever being shown
one. The bottom row is drawn from **content alone**; it is a clean, denoised canonical view, because noise cannot be
predicted from two numbers.

In [ ]:
# ── 4.5 Evaluate: content predictivity P(c|z) and pose recovery ──
z_h_tr, z_h_te = codes(harmony, X_TRAIN), codes(harmony, X_TEST)
with torch.no_grad():
    th_h = harmony.encode(X_TEST)[0].cpu().numpy()
RESULTS["Harmony"] = (predictivity(z_h_tr, z_h_te), few_label_accuracy(z_h_tr, Y_TRAIN, z_h_te, Y_TEST, 10))
plot_latent(z_h_te, Y_TEST, TH_TEST, "Harmony")
print(f"Harmony  P(c|z) = {RESULTS['Harmony'][0]:.3f}   10-label 1-NN = {RESULTS['Harmony'][1]:.3f}")
for period in (360, 180):
    e = pose_error_deg(th_h, TH_TEST, Y_TEST, period)
    per_class = ", ".join(f"{SHORT[c]} {np.median(e[Y_TEST == c]):.1f}°" for c in range(3))
    print(f"pose error mod {period}°: median {np.median(e):5.1f}° ({per_class}); within 15°: {np.mean(e < 15):.0%}")

fig, axes = plt.subplots(1, 3, figsize=(10, 3.1), sharey=True)
for c, ax in enumerate(axes):
    m = Y_TEST == c
    ax.scatter(np.degrees(TH_TEST[m]), np.degrees(np.angle(np.exp(1j * th_h[m]))), s=4)
    ax.set_title(CLASS_NAMES[c]); ax.set_xlabel("true angle θ (°)")
axes[0].set_ylabel(r"predicted normalising angle $\hat\theta$ (°)")
fig.suptitle(r"Harmony's pose: lines of slope −1 mean $\hat\theta = -\theta$ + const (two lines 180° apart = half-turn flips)",
             fontsize=9)
plt.tight_layout(); plt.show()
assert RESULTS["Harmony"][1] > RESULTS["raw pixels"][1] + 0.1, "Harmony's code should beat raw pixels with few labels"

🔍 **What just happened?** Harmony's content code separates the three molecules — including the proteasome and the
*capped* proteasome, which differ only by the cap — and the angle colouring is scrambled: **pose has been pushed out of
$z$**. With only 10 labelled particles per class, a nearest-neighbour classifier on $z$ is almost perfect, whereas raw
pixels reach only about 60 % (chance: 33 %). The predicted angle follows $\hat\theta = -\theta + \text{const}$: Harmony recovered
the pose with a few degrees of error *modulo a half turn* (some particles are normalised upside down — compare the two
rows of numbers above).

💡 Why "−θ"? $\hat\theta$ is the rotation that **undoes** the particle's pose, and the constant is the class's
arbitrary canonical orientation, which the network is free to choose.

### 4.6 Ablation: remove the Siamese branch

Without the copy $x'$, nothing tells the transformer to undo the pose: $D^{(1)}(x_z, x_k)$ alone is satisfied just as
well if the transformer does anything at all and $z$ encodes whatever pose remains. Let us check.

In [ ]:
# ── 4.6 Ablation: Harmony without the Siamese branch (only γ·D⁽¹⁾) ──
torch.manual_seed(SEED)
harmony_solo = Harmony(z_dim=2)
hist_solo = train(harmony_solo, lambda m, x: harmony_loss(m, x, siamese=False), X_TRAIN, steps=1000, name="no Siamese")
z_s_tr, z_s_te = codes(harmony_solo, X_TRAIN), codes(harmony_solo, X_TEST)
with torch.no_grad():
    th_solo = harmony_solo.encode(X_TEST)[0].cpu().numpy()
RESULTS["Harmony, no Siamese"] = (predictivity(z_s_tr, z_s_te), few_label_accuracy(z_s_tr, Y_TRAIN, z_s_te, Y_TEST, 10))
e_solo = pose_error_deg(th_solo, TH_TEST, Y_TEST, 180)
print(f"no Siamese: pose error mod 180°: median {np.median(e_solo):.1f}°, within 15°: {np.mean(e_solo < 15):.0%}")
plot_latent(z_s_te, Y_TEST, TH_TEST, "no Siamese")

In [ ]:
# ── 4.7 Summary: which code knows what the molecule is? ──
names = list(RESULTS)
vals = np.array([RESULTS[n] for n in names])
fig, ax = plt.subplots(figsize=(7.5, 3.2))
xpos = np.arange(len(names))
ax.bar(xpos - 0.2, vals[:, 0], 0.4, label="P(c|z): 5-NN, all training codes")
ax.bar(xpos + 0.2, vals[:, 1], 0.4, label="1-NN with only 10 labels per class")
ax.axhline(1 / 3, color="gray", ls=":", lw=1); ax.text(len(names) - 0.5, 0.35, "chance", fontsize=7, color="gray")
ax.set_xticks(xpos); ax.set_xticklabels(names, fontsize=8); ax.set_ylim(0, 1.05); ax.set_ylabel("test accuracy")
ax.legend(fontsize=7, loc="lower right"); ax.set_title("Class information in each representation (held-out test set)")
plt.tight_layout(); plt.show()
print(f"{'representation':22s} {'P(c|z)':>7s} {'10 labels':>10s}")
for n, (a, b) in RESULTS.items():
    print(f"{n:22s} {a:7.2f} {b:10.2f}")
assert RESULTS["Harmony"][1] > RESULTS["Harmony, no Siamese"][1], "the Siamese branch should matter"

🔍 **What just happened?** Removing the Siamese branch collapses the result: the transformer no longer normalises the
pose, the code becomes pose-dominated again, and few-label accuracy drops towards that of raw pixels. The Siamese
**consistency** — "two copies of one particle must look identical after normalisation" — is what teaches the network
what *pose* means. Notice also the left bars: when every pose is densely covered by hundreds of labelled examples, even raw
pixels give a high $P(c\,|\,z)$; the difference shows up when labels are scarce, which is the realistic case.

✏️ **Your turn.** (a) Re-run Sections 1.2–4.7 with `SNR = 0.3`. Which representation degrades most? (b) Harmony's paper
used a **one-dimensional** content code on real data; try `Harmony(z_dim=1)` in 4.3 (the latent plots then show $z_1$
only on one axis — adapt `plot_latent` or just read the printed numbers).

<details><summary>Hints</summary>At lower SNR all methods lose accuracy and pose estimates get noisier; Harmony usually
keeps the largest margin over raw pixels in the 10-label column, but needs more training steps. With $z_{\dim}=1$ the
three classes must line up along one axis — check $P(c\,|\,z)$.</details>

### 📄 Paper connection — Harmony

**Uddin, Howe, Zeng & Xu (2022).** *Harmony: A Generic Unsupervised Approach for Disentangling Semantic Content From
Parameterized Transformations.* CVPR 2022, pp. 20646–20655.
[CVF open access](https://openaccess.thecvf.com/content/CVPR2022/papers/Uddin_Harmony_A_Generic_Unsupervised_Approach_for_Disentangling_Semantic_Content_From_CVPR_2022_paper.pdf)
- **Question it answers:** can we learn, without labels, a content code that ignores pose (rotation, shift, scale,
  contrast) while also predicting the pose, for cryo-EM/ET particles and natural images?
- **What the paper does:** encoder → $(k_x, \psi_x)$; spatial transformer $x_k = T(x; k_x)$; content-only decoder;
  Siamese copy under a random known transformation ("cross-contrastive learning"); loss
  $\gamma[D^{(1)}(x_z,x_k)+D^{(2)}(x_k,x'_{k'})+D^{(3)}(x'_{k'},x'_{z'})]+D_{KL}(P_\psi\Vert P_{\psi'})$ with SSE distances,
  $\gamma\propto M/N$, and no direct $D^{(4)}(x_z,x'_{z'})$ term. 2D models (MLP encoder/decoder) for MNIST, two
  negative-stain EM data sets and CelebA (contrast); a 3D convolutional model (32³ subtomograms; 3 angles + 3 shifts via an
  affine matrix — the paper does not name an Euler convention).
- **What our toy keeps:** all four ingredients and the exact loss structure; MLP encoder/decoder; the code's random-known
  transform ranges (±90°, ±3 px) and $\gamma=\lceil M/(1000N)\rceil$; kNN predictivity $P(c\,|\,z)$.
- **What it leaves out:** 3D, scale/contrast parameters, the SAP disentanglement score, real data, CTF and missing
  wedge, the paper's network sizes and training length; we bound shifts with tanh (the paper says parameters are bounded
  by activation functions; the released 2D code does not bound them) and use a 2-D (not 1-D) content code for plotting.
- **Result reported:** on MNIST the authors report $P(c\,|\,z)=0.944$ (SAP 0.55). On cryo-ET subtomograms (Harmony vs 3D
  spatial VAE, SAP / $P(c|z)$): simulated SNR 100: 0.494/0.996 vs 0.409/0.997; SNR 0.1: 0.384/0.861 vs 0.382/0.858; and on a
  **real rat-neuron tomogram, Harmony's one-dimensional content code separated ribosomes from proteasomes with
  $P(c\,|\,z) = 0.999$ (SAP 0.268) versus 0.69 (SAP 0.011) for the 3D spatial VAE**. Both methods perform poorly at
  SNR ≤ 0.03. Alignment is shown only qualitatively.
- **Code:** [`aitom/classify/deep/unsupervised/disentangled_representation/harmony`](https://github.com/xulabs/aitom/tree/master/aitom/classify/deep/unsupervised/disentangled_representation/harmony)
  (`model2D`, `model3D`, `modelColor`; loss in `model2D/utils.py::loss_fn`).

---
## 5 · Beyond Harmony: compositions, conformations and multiple choices

Harmony describes pose with a few **explicit** parameters (angle, shift, scale, contrast). Two newer works from the lab
continue this line. We could **not** read either while writing this notebook, so we give only what can be checked.

### 📄 Paper connection — successors (⚠️ content not verified)

**Uddin, Armouti & Xu (2025).** *Unsupervised Identification of Protein Compositions and Conformations via Implicit
Content-Transformation Disentanglement.* ICCV 2025.
> ⚠️ We could not access the full text of this paper while writing; the description below is limited to its title and
> venue (from the lab's publication list). Read the paper before relying on details.
- **General goal (from the title only):** identify protein *compositions* (e.g. capped vs uncapped) and *conformations*
  without labels, by disentangling content from transformation — with the transformation represented **implicitly**
  rather than by a fixed list of parameters. This is a reading of the title, not a description of the method.

**DUIL — Deep unsupervised inverse learning for in situ macromolecular morphology identification** (listed as NeurIPS 2026).
> ⚠️ We could not find or read this paper while writing (no public record was found); only the title is given. Read the
> paper before relying on details.

**Related but distinct (verified abstract):** Uddin, Vora, Wu, Chen & Xu, *Unsupervised SE(3) Disentanglement for in
situ Macromolecular Morphology Identification from Cryo-Electron Tomography* — an **ICLR 2026 submission** (OpenReview
`AyX26Srw0r`, recorded as rejected). We list it as *related work*, **not** as DUIL. Its abstract treats a morphology as a
denoised, SE(3)-invariant volume (SE(3) = 3D rotations + translations), so identifying it means jointly estimating a
template and each particle's pose; it argues that maximum-likelihood subtomogram averaging misses rare morphologies and
needs manual tuning, and proposes a disentangled framework that separates SE(3) pose from morphology content, with a
**multi-choice learning** module for high noise; the learned content generates templates.

### 5.1 Toy: multiple-choice (winner-takes-all) learning

**Multiple-choice learning** (Guzman-Rivera *et al.*, 2012) trains $K$ hypotheses instead of one, and for each example
only the **best** hypothesis is penalised ("winner takes all"):

$$\mathcal{L}_{\text{WTA}} = \frac{1}{N}\sum_{i=1}^{N} \min_{k=1..K} \big\lVert x_i - h_k \big\rVert^2 .$$

*In words:* each hypothesis specialises on the examples it explains best, so a rare kind of particle can get its own
hypothesis instead of being averaged away. Here the hypotheses are simply $K$ learnable **canonical images** fitted to
Harmony's pose-normalised images $x_k$ of an *imbalanced* population (many ribosomes, few capped proteasomes). With
images as hypotheses the WTA loss is exactly the k-means objective — the same "class averaging" as in Module 14; in the
research papers the hypotheses are neural decoders. ⚠️ This is an **illustration of multiple-choice learning, not the
algorithm of the papers above.**

In [ ]:
# ── 5.1 Winner-takes-all hypotheses on Harmony's pose-normalised images ──
with torch.no_grad():
    th_all, sh_all, _, _ = harmony.encode(X_TEST)
    XK = spatial_transform(X_TEST, th_all, sh_all).flatten(1).cpu()
g = np.random.default_rng(SEED)
population = {0: 160, 1: 60, 2: 20}                  # ribosomes : proteasomes : rare capped proteasomes
sel = np.concatenate([g.choice(np.where(Y_TEST == c)[0], n, replace=False) for c, n in population.items()])
X_wta, y_wta = XK[sel], Y_TEST[sel]


def wta_fit(X, K, steps=300, lr=0.05, seed=SEED):
    """Multiple-choice learning with K learnable canonical images and a winner-takes-all loss.

    The hypotheses start as K randomly chosen examples. Returns (hypotheses [K, P], index of the winning hypothesis
    for every example).
    """
    gen = torch.Generator().manual_seed(seed)
    H = X[torch.randperm(len(X), generator=gen)[:K]].clone().requires_grad_(True)
    opt = torch.optim.Adam([H], lr=lr)
    for _ in range(steps):
        err = ((X[:, None, :] - H[None]) ** 2).sum(-1)          # [N, K] squared error of every hypothesis
        loss = err.min(1).values.mean()                          # only the winner is penalised
        opt.zero_grad(); loss.backward(); opt.step()
    return H.detach(), err.argmin(1).numpy()


fig, axes = plt.subplots(1, 8, figsize=(9.6, 1.95))
H1, _ = wta_fit(X_wta, 1)
axes[0].imshow(H1[0].view(N_PIX, N_PIX)); axes[0].set_title("K = 1\n(one decoder)", fontsize=7)
tables = {}
for j, K in enumerate([3, 4]):
    HK, assign = wta_fit(X_wta, K)
    tables[K] = np.array([[np.sum((assign == k) & (y_wta == c)) for c in range(3)] for k in range(K)])
    for k in range(K):
        ax = axes[1 + k + 3 * j]
        ax.imshow(HK[k].view(N_PIX, N_PIX)); ax.set_title(f"K={K}: #{k} ({np.sum(assign == k)} wins)", fontsize=7)
for ax in axes:
    ax.axis("off")
fig.suptitle("Winner-takes-all hypotheses on pose-normalised images (population 160 : 60 : 20)", fontsize=9)
plt.tight_layout(); plt.show()
for K, tab in tables.items():
    print(f"K = {K}: rows = hypotheses, columns = true molecules ({', '.join(SHORT)})")
    for k, row in enumerate(tab):
        print(f"   hypothesis #{k}: {row.tolist()}")

🔍 **What just happened?** One hypothesis ($K=1$) must explain every particle at once, so it becomes a blurred mixture —
exactly what a single decoder does to a rare species. With more hypotheses they specialise into denoised canonical views.
Read the tables: the ribosomes claim two hypotheses, because they come in **two** pose-normalised versions — upright
and flipped by 180° (the half-turn ambiguity of Section 4.5). With $K = 3$ one of those two also absorbs both
proteasomes, and the third hypothesis wins only a handful of particles (an almost "dead" hypothesis). With $K = 4$ the
20S proteasome gets a hypothesis of its own, but the rare capped proteasome still has none — it hides inside a ribosome
hypothesis: extra hypotheses go first to the most abundant, most variable species, not to the rare one that differs
only by a cap. Multiple choices *can* make room for rare states — the ✏️ below explores when — but
they also expose the failure modes of the pose model. *Choosing K* is a
modelling decision (Modules 04 and 22 used BIC for that).

✏️ **Your turn.** Make the capped proteasome rarer (`population = {0: 160, 1: 60, 2: 6}`) and re-run 5.1. Does it still
get its own hypothesis with $K=4$? Try `seed=1` in `wta_fit`: winner-takes-all is sensitive to where the hypotheses start
(some may never win — "dead" hypotheses, a known weakness that relaxed variants address).

### From 2D toys to subtomograms

| | our 2D toy | Harmony 3D (paper/code) |
|---|---|---|
| input | 32×32 projection | 32³ subtomogram |
| pose $k$ | 1 angle + 2 shifts | 3 rotation angles + 3 shifts (affine matrix) |
| encoder | MLP | 3D convolutions (4³ kernels, stride 2; 32→64→128→512 channels) |
| random known transform | ±90°, ±3 px | angles ±90°, shifts ±4 voxels |
| data | simulated, SNR 1 | simulated SNR 100…0.01, and a real rat-neuron tomogram |

Real tomograms add the **missing wedge** (Module 07): the same particle looks different depending on how it is
oriented relative to the beam, which is part of what makes 3D disentanglement much harder than our toy.
For *continuous* conformational change in single-particle cryo-EM, see the deep-dive tutorial on CryoDRGN in
[`../../HeterogeneousReconstruction/`](../../HeterogeneousReconstruction/).

---
## ✅ Checkpoint

1. Why does 1-NN on raw pixels need many labels per class, while 1-NN on oracle-aligned pixels needs very few?
   <details><summary>Answer</summary>Raw-pixel distance is dominated by pose: the nearest labelled image is one turned the
   same way, whatever the molecule. Only when every pose of every class is covered by some labelled example does 1-NN work.
   Removing the pose leaves content differences, which a few examples already capture.</details>
2. What does the reparameterization trick do, and why is it needed?
   <details><summary>Answer</summary>It writes a random sample as $z = \mu + \sigma\varepsilon$ with
   $\varepsilon\sim\mathcal N(0,1)$, so the randomness is an input and gradients can flow through $\mu$ and $\sigma$ to the
   encoder. Sampling $z$ directly from $\mathcal N(\mu,\sigma^2)$ would block backpropagation.</details>
3. In a spatial VAE, why does changing $\theta$ always give an exact rotation of the drawn molecule?
   <details><summary>Answer</summary>Because $\theta$ only enters the decoder through the coordinates, $g(R(\theta)u + t, z)$:
   the same function is evaluated on rotated coordinates, so the output is the $\theta=0$ drawing rotated (by $-\theta$ in our
   convention).</details>
4. Which Harmony loss term forces the transformer to undo the pose consistently, and what happens without it?
   <details><summary>Answer</summary>$D^{(2)}(x_k, x'_{k'})$ (together with the KL between the two content posteriors):
   two copies of one particle in different poses must give the same pose-normalised image. Without the Siamese branch
   (our ablation) pose leaks back into $z$ and few-label accuracy collapses.</details>
5. Harmony predicts $\hat\theta \approx -\theta + \text{const}$. Why minus, and why a constant?
   <details><summary>Answer</summary>$\hat\theta$ is the rotation that *undoes* the particle's rotation $\theta$; the constant
   is the canonical orientation of that class, which is arbitrary and chosen by the network.</details>
6. What do we actually know about the ICCV 2025 and DUIL papers from this notebook?
   <details><summary>Answer</summary>Only their titles and venues (and, separately, the abstract of a related ICLR 2026
   submission). Their methods and results were not verified — read the papers.</details>

## 🧠 Questions & discussion

1. Harmony's content code separated ribosomes from proteasomes on real data. What would you need to check before claiming
   it can separate two *conformations* of the ribosome?
2. The capped and uncapped proteasome share a whole barrel. Which loss term would be most affected if the cap were very
   small (a few pixels)? How could you test that with this notebook?
3. We saw half-turn flips in the recovered poses. For which biological molecules would a "flip" be harmless, and for which
   would it corrupt a subtomogram average?
4. The authors deliberately left out a direct $D^{(4)}(x_z, x'_{z'})$ term. Propose an experiment to test their claim that it
   hinders escaping local minima.
5. A disentangled code is learned without labels. How could it still be *biased*, e.g. by the composition of the training
   set or by the choice of allowed transformations?

## 🚀 Bonus — going further

- **Real code.** Run the official 2D Harmony (`main-2D.py` in the AITom
  [`harmony`](https://github.com/xulabs/aitom/tree/master/aitom/classify/deep/unsupervised/disentangled_representation/harmony)
  folder) on MNIST and compare its latent manifold with ours. Note that it uses `kornia` for the transformer.
- **Angles without wrap-around.** Predict $(\cos\theta, \sin\theta)$ with two outputs and use `atan2`; does it reduce the
  half-turn flips? (Our quick tests suggest it changes the failure mode rather than removing it — measure it.)
- **Go 3D.** Replace the 2D views by 24³ subtomograms (toolkit `simulate_subtomogram`) and the transformer by
  `F.affine_grid` in 3D with three angles.
- **SAP score.** Implement the SAP disentanglement score used in the paper and compare it with $P(c\,|\,z)$.
- **Read** the Harmony paper's Section on "cross-contrastive learning" and the Kim *et al.* (2023) review's discussion of
  unbiased pattern mining (Module 00 reference list).

## 📝 Lab note

```text
Module 23 — date:
Input: 2D side views of 4V4R / 3J9I / 6UTJ, random pose (±180°, ±3 px), SNR 1
Output: content codes z, pose estimates θ̂, pose-normalised images, WTA hypotheses
Method: VAE, spatial VAE, Harmony-lite (Siamese, 3×SSE + KL), ablation without Siamese
Key parameter changed & effect:
Failure mode observed (e.g. half-turn flips, pose leaking into z):
Paper link: Uddin et al. CVPR 2022 (Harmony); Yi et al. arXiv 2021
```

## 📚 References

1. Uddin MR, Howe G, Zeng X, Xu M. Harmony: A generic unsupervised approach for disentangling semantic content from parameterized transformations. *CVPR* 2022, pp. 20646–20655.
2. Yi K, Pang J, Zhang Y, Zeng X, Xu M. Disentangling semantic features of macromolecules in Cryo-Electron Tomography. arXiv:2106.14192 (2021).
3. Uddin MR, Armouti J, Xu M. Unsupervised identification of protein compositions and conformations via implicit content-transformation disentanglement. *ICCV* 2025. ⚠️ content not verified.
4. DUIL: Deep unsupervised inverse learning for in situ macromolecular morphology identification. *NeurIPS* 2026 (as listed by the lab). ⚠️ citation and content not verified.
5. Uddin MR, Vora MB, Wu Q, Chen M, Xu M. Unsupervised SE(3) disentanglement for in situ macromolecular morphology identification from cryo-electron tomography. ICLR 2026 submission, OpenReview `AyX26Srw0r` (rejected; related work).
6. Bepler T, Zhong ED, Kelley K, Brignole E, Berger B. Explicitly disentangling image content from translation and rotation with spatial-VAE. *NeurIPS* 2019. arXiv:1909.11663.
7. Kingma DP, Welling M. Auto-encoding variational Bayes. *ICLR* 2014. arXiv:1312.6114.
8. Jaderberg M, Simonyan K, Zisserman A, Kavukcuoglu K. Spatial transformer networks. *NeurIPS* 2015. arXiv:1506.02025.
9. Guzman-Rivera A, Batra D, Kohli P. Multiple choice learning: learning to produce multiple structured outputs. *NeurIPS* 2012.
10. Kim HH, Uddin MR, Xu M, Chang YW. Computational methods toward unbiased pattern mining and structure determination in cryo-electron tomography data. *J Mol Biol* 435:168068 (2023). doi:10.1016/j.jmb.2023.168068

## ➡️ Next

**[24 · Denoising without clean data: J-invariance, cycle consistency and score matching](24_denoising_without_clean_data.ipynb)**
— Harmony's decoder produced clean pictures as a side effect; next we make *denoising* the goal, without ever seeing a
clean tomogram. Previous: [22 · Sorting a cell's particles without labels](22_unsupervised_clustering_disca.ipynb).
Course overview: [`README.md`](../README.md) · [`GLOSSARY.md`](../GLOSSARY.md) · [`PAPER_ATLAS.md`](../PAPER_ATLAS.md).